# APIs & Networking for Data Engineers
## REST, GraphQL, gRPC, Authentication & Building Data Products

**What you'll learn:**
- REST API fundamentals: methods, status codes, headers, pagination
- Authentication: API keys, OAuth 2.0, JWT tokens, service accounts
- Pagination strategies: offset, cursor, keyset
- Rate limiting: handling 429s, exponential backoff, throttling
- GraphQL: when it's better than REST for data ingestion
- gRPC & Protocol Buffers: high-performance inter-service communication
- Webhooks: event-driven data ingestion
- Building data APIs: exposing data products to consumers
- Reverse ETL: pushing warehouse data to SaaS tools
- API design best practices for data platforms
- Common ingestion patterns with Python (requests, aiohttp)
- Interview questions

**Estimated Time:** 6-8 hours

---

> As a DE, you're both a CONSUMER of APIs (ingesting data from SaaS tools)
> and a PROVIDER of APIs (exposing data products to other teams).
> Understanding both sides makes you infinitely more effective.

---

---
# Section 1: REST API Fundamentals

## HTTP Methods (What You Actually Use as DE)

| Method | Purpose | Idempotent? | DE Use Case |
|--------|---------|-------------|-------------|
| **GET** | Read/retrieve data | Yes | Fetch records from API |
| **POST** | Create or submit data | No | Send events, trigger jobs |
| **PUT** | Replace entirely | Yes | Update a full record |
| **PATCH** | Partial update | No | Update specific fields |
| **DELETE** | Remove resource | Yes | Delete records, cleanup |

## Status Codes You Must Know

```
2xx SUCCESS:
  200 OK           -- Request succeeded (most common response)
  201 Created      -- Resource created (POST success)
  202 Accepted     -- Async processing started (job submitted)
  204 No Content   -- Success but no response body (DELETE)

3xx REDIRECT:
  301 Moved        -- Use new URL permanently
  302 Found        -- Temporary redirect
  304 Not Modified -- Use cached version (ETag match)

4xx CLIENT ERROR (your fault):
  400 Bad Request  -- Malformed request (check your payload)
  401 Unauthorized -- Authentication failed (bad/expired token)
  403 Forbidden    -- Authenticated but no permission
  404 Not Found    -- Resource doesn't exist
  409 Conflict     -- Concurrent modification conflict
  422 Unprocessable-- Valid syntax but semantic error
  429 Too Many Requests -- RATE LIMITED (back off!)

5xx SERVER ERROR (their fault):
  500 Internal Error   -- Server bug (retry may help)
  502 Bad Gateway      -- Upstream server issue
  503 Service Unavail  -- Overloaded (retry with backoff)
  504 Gateway Timeout  -- Upstream timeout (retry)
```

## Headers That Matter for DE

| Header | Purpose | Example |
|--------|---------|---------|
| `Authorization` | Authentication token | `Bearer eyJhbG...` |
| `Content-Type` | Request body format | `application/json` |
| `Accept` | Desired response format | `application/json` |
| `X-RateLimit-Remaining` | Requests left in window | `47` |
| `X-RateLimit-Reset` | When limit resets (epoch) | `1719100800` |
| `ETag` / `If-None-Match` | Caching / conditional requests | Avoid re-downloading unchanged data |
| `Retry-After` | How long to wait (on 429/503) | `60` (seconds) |

In [ ]:
# REST API patterns for data ingestion
print("REST API INGESTION PATTERNS")
print("=" * 60)

print("""
# PATTERN 1: Basic paginated extraction (most common DE task)

import requests
import time

def extract_all_records(base_url, api_key, page_size=100):
    """Extract all records from a paginated REST API."""
    session = requests.Session()
    session.headers.update({
        "Authorization": f"Bearer {api_key}",
        "Accept": "application/json",
    })
    
    all_records = []
    page = 1
    
    while True:
        response = session.get(
            f"{base_url}/orders",
            params={"page": page, "per_page": page_size},
            timeout=30,
        )
        
        # Handle rate limiting
        if response.status_code == 429:
            retry_after = int(response.headers.get("Retry-After", 60))
            print(f"Rate limited. Waiting {retry_after}s...")
            time.sleep(retry_after)
            continue
        
        response.raise_for_status()  # Raise on 4xx/5xx
        data = response.json()
        
        records = data.get("results", [])
        if not records:
            break
            
        all_records.extend(records)
        page += 1
        
        # Respect rate limits proactively
        remaining = int(response.headers.get("X-RateLimit-Remaining", 100))
        if remaining < 5:
            time.sleep(1)  # Slow down before hitting limit
    
    return all_records


# PATTERN 2: Cursor-based pagination (better for large datasets)

def extract_with_cursor(base_url, api_key):
    """Cursor pagination -- no skipping, no duplicates."""
    session = requests.Session()
    session.headers["Authorization"] = f"Bearer {api_key}"
    
    cursor = None
    while True:
        params = {"limit": 200}
        if cursor:
            params["cursor"] = cursor
        
        response = session.get(f"{base_url}/events", params=params)
        response.raise_for_status()
        data = response.json()
        
        yield data["results"]  # Generator -- memory efficient!
        
        cursor = data.get("next_cursor")
        if not cursor:
            break


# PATTERN 3: Incremental extraction (only new/changed records)

def extract_incremental(base_url, api_key, last_modified_after):
    """Only fetch records modified since last run."""
    params = {
        "modified_after": last_modified_after.isoformat(),
        "sort": "modified_at",
        "order": "asc",
    }
    # ... (same pagination logic)
    # After success, save the latest modified_at as checkpoint
    # Next run starts from that checkpoint
""")

print("KEY PRINCIPLES:")
print("  1. Always handle 429 (rate limit) with exponential backoff")
print("  2. Use cursor pagination when available (no duplicates/skips)")
print("  3. Incremental extraction (only new data since last run)")
print("  4. Use sessions (reuses TCP connections = faster)")
print("  5. Set timeouts (never hang indefinitely)")
print("  6. Log: total records, pages, errors, duration")

---
# Section 2: Authentication & Authorization

## Authentication Methods for API Ingestion

| Method | Security | Use Case | Example |
|--------|----------|----------|---------|
| **API Key** | Low-Medium | Simple SaaS APIs | `X-API-Key: sk_live_abc123` |
| **OAuth 2.0 (Client Credentials)** | High | Service-to-service | Machine-to-machine, no user |
| **OAuth 2.0 (Authorization Code)** | High | On behalf of user | User-consented access |
| **JWT Bearer Token** | High | Stateless auth | Short-lived, self-contained |
| **AWS IAM (SigV4)** | Very High | AWS service access | Signed requests with access key |
| **mTLS** | Very High | Zero-trust networks | Mutual certificate verification |

## OAuth 2.0 Client Credentials Flow (Most Common for DE)

```
YOUR PIPELINE                    AUTH SERVER               TARGET API
┌──────────┐                   ┌──────────────┐         ┌──────────┐
│          │ 1. POST /token    │              │         │          │
│          │ (client_id +      │              │         │          │
│  Data    │  client_secret)   │   OAuth      │         │  Source  │
│  Pipeline│──────────────────>│   Provider   │         │  API     │
│          │                   │ (Okta, Auth0)│         │(Salesforce│
│          │<──────────────────│              │         │  Stripe) │
│          │ 2. access_token   │              │         │          │
│          │    (expires 1hr)  │              │         │          │
│          │                   └──────────────┘         │          │
│          │                                            │          │
│          │ 3. GET /data                               │          │
│          │    Authorization: Bearer <token>           │          │
│          │───────────────────────────────────────────>│          │
│          │<───────────────────────────────────────────│          │
│          │ 4. Response data                           │          │
└──────────┘                                            └──────────┘

DE Implementation:
  - Store client_id + client_secret in AWS Secrets Manager
  - Token refresh: get new token when current expires (or 401 received)
  - Cache token until expiry (don't request new token every API call!)
```

## Secrets Management for Pipelines

```
NEVER DO THIS:
  api_key = "sk_live_abc123def456"  # Hardcoded in code!

DO THIS:
  # Option 1: Environment variable (simple)
  api_key = os.environ["STRIPE_API_KEY"]
  
  # Option 2: AWS Secrets Manager (production)
  import boto3
  client = boto3.client('secretsmanager')
  secret = client.get_secret_value(SecretId='stripe/api_key')
  api_key = json.loads(secret['SecretString'])['key']
  
  # Option 3: Databricks Secret Scopes
  api_key = dbutils.secrets.get(scope="apis", key="stripe_key")
  
  # Option 4: Airflow Connections
  from airflow.hooks.base import BaseHook
  conn = BaseHook.get_connection("stripe_api")
  api_key = conn.password
```

In [ ]:
# Pagination strategies comparison
print("PAGINATION STRATEGIES")
print("=" * 60)

print("""
1. OFFSET PAGINATION (simplest, worst for large data):
   GET /orders?page=1&per_page=100
   GET /orders?page=2&per_page=100
   
   Problems:
     - page=10000 is SLOW (DB must skip 999,900 rows)
     - If records are inserted between pages, you get DUPLICATES or SKIPS
   
   Use when: small datasets, simple APIs, < 10K total records

2. CURSOR PAGINATION (recommended):
   GET /orders?limit=100
   Response: {"results": [...], "next_cursor": "eyJpZCI6MTAwfQ=="}
   GET /orders?limit=100&cursor=eyJpZCI6MTAwfQ==
   
   Benefits:
     - Consistent (no duplicates/skips even with concurrent inserts)
     - Fast (DB seeks to cursor position directly)
     - No "total count" needed
   
   Use when: large datasets, Stripe/Shopify/GitHub APIs

3. KEYSET PAGINATION (best performance):
   GET /orders?limit=100&created_after=2024-06-01T00:00:00Z
   Response: last record has created_at=2024-06-01T12:34:56Z
   GET /orders?limit=100&created_after=2024-06-01T12:34:56Z
   
   Benefits:
     - Fastest (uses index on sort column)
     - Works with incremental extraction naturally
   
   Use when: you control the API, time-series data

4. LINK-HEADER PAGINATION (GitHub style):
   Response headers:
   Link: <https://api.github.com/repos?page=2>; rel="next",
         <https://api.github.com/repos?page=10>; rel="last"
   
   Just follow the "next" link until there isn't one.
""")

# Rate limiting handling
print("\nRATE LIMITING -- EXPONENTIAL BACKOFF:")
print("""
import time
import random

def call_with_backoff(func, max_retries=5):
    for attempt in range(max_retries):
        try:
            response = func()
            if response.status_code == 429:
                # Use Retry-After header if available
                wait = int(response.headers.get("Retry-After", 2 ** attempt))
                # Add jitter to prevent thundering herd
                wait += random.uniform(0, 1)
                print(f"Rate limited. Waiting {wait:.1f}s (attempt {attempt+1})")
                time.sleep(wait)
                continue
            return response
        except (ConnectionError, Timeout) as e:
            wait = 2 ** attempt + random.uniform(0, 1)
            print(f"Error: {e}. Retry in {wait:.1f}s")
            time.sleep(wait)
    raise Exception(f"Failed after {max_retries} retries")
""")

---
# Section 4: GraphQL & gRPC

## GraphQL for Data Ingestion

```
REST problem: Over-fetching and under-fetching
  GET /users/123         → returns ALL 50 fields (you need 3)
  GET /users/123/orders  → separate request for orders
  GET /users/123/address → separate request for address
  = 3 API calls, tons of wasted bandwidth

GraphQL solution: ONE request, get EXACTLY what you need
  query {
    user(id: "123") {
      name
      email
      orders(last: 5) {
        id
        amount
        status
      }
    }
  }
  = 1 API call, only requested fields returned
```

## When to Use GraphQL vs REST for DE

| Factor | REST | GraphQL |
|--------|------|---------|
| Pagination | Built-in (cursor/offset) | Connections pattern |
| Rate limiting | Standard (per endpoint) | Complex (query cost) |
| Caching | Easy (HTTP caching) | Hard (POST requests) |
| Bulk extraction | Good (list endpoints) | Can be slow (N+1) |
| Nested data | Multiple requests | Single request |
| Schema discovery | OpenAPI/Swagger | Introspection query |
| Best for DE | Most ingestion tasks | APIs with nested relationships |

## gRPC & Protocol Buffers

```
gRPC: High-performance RPC (Remote Procedure Call) framework
  - 10x faster than REST (binary protocol, HTTP/2)
  - Strongly typed (Protocol Buffers schema)
  - Bidirectional streaming
  - Used by: Kafka internal communication, Kubernetes, many microservices

When DE encounters gRPC:
  - Internal microservices exposing data via gRPC
  - High-throughput streaming between services
  - ML model serving endpoints (TF Serving, Triton)
  
Proto definition example:
  service OrderService {
    rpc GetOrders (OrderRequest) returns (stream OrderResponse);
    rpc StreamEvents (google.protobuf.Empty) returns (stream Event);
  }
  
  message OrderResponse {
    int64 order_id = 1;
    string customer_id = 2;
    double amount = 3;
    google.protobuf.Timestamp created_at = 4;
  }
```

## Webhooks: Event-Driven Ingestion

```
POLLING (traditional):             WEBHOOKS (event-driven):
  Every 5 min: "Any new data?"       Source: "Here's new data!"
  Source: "No"                        
  Every 5 min: "Any new data?"       Your endpoint receives POST
  Source: "No"                        immediately when event occurs.
  Every 5 min: "Any new data?"
  Source: "Yes, here's 1 record"     Benefits:
                                       - Real-time (no delay)
  Wastes: 99% of API calls            - Efficient (no wasted calls)
  Delay: up to 5 min                   - Lower API quota usage

WEBHOOK RECEIVER PATTERN:
  1. Expose an endpoint: POST /webhooks/stripe
  2. Verify signature (prevent spoofing)
  3. Acknowledge immediately (200 OK within 5s)
  4. Process async (queue the event for processing)
  5. Handle duplicates (idempotent processing)
  6. Handle retries (webhook providers retry on failure)
```

---
# Section 5: Building Data APIs & Reverse ETL

## Data APIs: Exposing Data Products

```
Your data warehouse/lake contains valuable computed data.
Other teams need access to it. Options:

1. DIRECT TABLE ACCESS (Unity Catalog sharing):
   - Grant SELECT on gold.customer_metrics
   - Best for: internal analytics, BI tools

2. REST API (for applications):
   - Expose computed data via API endpoints
   - Best for: applications needing real-time lookups
   
   GET /api/v1/customers/123/metrics
   Response: {
     "customer_id": "123",
     "lifetime_value": 15420.50,
     "churn_risk_score": 0.23,
     "segment": "enterprise",
     "last_order_date": "2024-06-10"
   }

3. REVERSE ETL (push data to SaaS tools):
   - Send computed segments/scores BACK to operational tools
   - CRM gets churn scores, marketing gets segments
   - Tools: Census, Hightouch, custom Airflow jobs
```

## Reverse ETL Architecture

```
DATA WAREHOUSE                    REVERSE ETL              DESTINATION
┌──────────────┐                ┌──────────────┐         ┌──────────┐
│ gold.customer│                │              │         │Salesforce│
│ _segments    │──── query ────>│  Census /    │── API ─>│(CRM)     │
│              │                │  Hightouch / │         └──────────┘
│ columns:     │                │  Custom      │         ┌──────────┐
│ - customer_id│                │              │── API ─>│HubSpot   │
│ - segment    │                │ Compares:    │         │(Marketing│
│ - ltv_score  │                │ current vs   │         └──────────┘
│ - churn_risk │                │ last sync    │         ┌──────────┐
│              │                │ Only sends   │── API ─>│Intercom  │
└──────────────┘                │ CHANGES      │         │(Support) │
                                └──────────────┘         └──────────┘

Pattern:
  1. Query warehouse for current state (e.g., customer segments)
  2. Compare with last synced state (track what changed)
  3. Push only CHANGED records to destination APIs
  4. Log: records synced, failures, latency
  5. Handle API rate limits and failures gracefully
```

## API Design Principles for Data Products

| Principle | Implementation |
|-----------|---------------|
| **Versioning** | `/api/v1/metrics` (never break existing consumers) |
| **Pagination** | Cursor-based by default, limit max page size |
| **Filtering** | Allow date ranges, segments, specific IDs |
| **Rate limiting** | Per-consumer quotas, return X-RateLimit headers |
| **Caching** | ETag for unchanged data, TTL for freshness |
| **Documentation** | OpenAPI/Swagger spec, auto-generated |
| **Authentication** | OAuth 2.0 or API keys with scopes |
| **Monitoring** | Latency p50/p95/p99, error rate, consumers |

In [ ]:
# API integration patterns for DE
print("API INTEGRATION PATTERNS")
print("=" * 60)

print("""
PATTERN 1: ASYNC BATCH INGESTION (for large APIs)

import asyncio
import aiohttp

async def fetch_page(session, url, page):
    async with session.get(url, params={"page": page}) as resp:
        return await resp.json()

async def extract_all_async(base_url, total_pages):
    async with aiohttp.ClientSession() as session:
        tasks = [fetch_page(session, base_url, p) for p in range(1, total_pages+1)]
        # Fetch 10 pages concurrently (respect rate limits!)
        results = []
        for batch in chunked(tasks, 10):
            batch_results = await asyncio.gather(*batch)
            results.extend(batch_results)
            await asyncio.sleep(1)  # Rate limit pause between batches
    return results

# 10x faster than sequential for I/O-bound API calls!


PATTERN 2: WEBHOOK RECEIVER (FastAPI)

from fastapi import FastAPI, Request, HTTPException
import hmac, hashlib

app = FastAPI()

@app.post("/webhooks/stripe")
async def receive_stripe_webhook(request: Request):
    # 1. Verify signature
    payload = await request.body()
    signature = request.headers.get("Stripe-Signature")
    if not verify_signature(payload, signature, WEBHOOK_SECRET):
        raise HTTPException(status_code=401, detail="Invalid signature")
    
    # 2. Parse event
    event = json.loads(payload)
    
    # 3. Write to queue for async processing (don't process inline!)
    await write_to_kafka("stripe_events", event)
    
    # 4. Acknowledge immediately
    return {"received": True}


PATTERN 3: DATA API WITH CACHING (FastAPI + Redis)

from fastapi import FastAPI
import redis

app = FastAPI()
cache = redis.Redis(host='localhost', port=6379)

@app.get("/api/v1/customers/{customer_id}/metrics")
async def get_customer_metrics(customer_id: str):
    # Check cache first
    cached = cache.get(f"metrics:{customer_id}")
    if cached:
        return json.loads(cached)
    
    # Query warehouse (expensive)
    metrics = query_warehouse(customer_id)
    
    # Cache for 5 minutes
    cache.setex(f"metrics:{customer_id}", 300, json.dumps(metrics))
    return metrics
""")

print("COMMON API INGESTION TOOLS:")
tools = [
    ("Fivetran", "Managed connectors (300+ sources)", "Zero-code, expensive"),
    ("Airbyte", "Open-source connectors", "Self-hosted or cloud"),
    ("Singer/Meltano", "Open-source ETL framework", "Tap + Target pattern"),
    ("Custom (requests)", "Python scripts", "Full control, more work"),
    ("Spark (REST source)", "DataFrame API for REST", "Batch extraction at scale"),
]
print(f"  {'Tool':<20} {'Description':<35} {'Notes'}")
print(f"  {'-'*70}")
for tool, desc, notes in tools:
    print(f"  {tool:<20} {desc:<35} {notes}")

In [ ]:
print("\nAPI & NETWORKING INTERVIEW QUESTIONS")
print("=" * 60)

questions = [
    {
        "q": "How would you design a pipeline to ingest data from a REST API with rate limits?",
        "a": "1) Use incremental extraction (only fetch new/changed since last_modified checkpoint). 2) Implement cursor pagination (not offset). 3) Respect rate limits: read X-RateLimit-Remaining headers, sleep when low. 4) Exponential backoff on 429/5xx with jitter. 5) Use connection pooling (requests.Session). 6) Log metrics: records fetched, pages, errors, duration. 7) Store checkpoints in a state table (last cursor/timestamp). 8) For high-volume: parallelize with async (aiohttp) within rate limits."
    },
    {
        "q": "What's the difference between polling and webhooks? When to use each?",
        "a": "Polling: your pipeline periodically asks 'any new data?' (pull model). Simple but wasteful and delayed. Webhooks: source pushes data to your endpoint when events occur (push model). Real-time, efficient, but requires: exposed endpoint, signature verification, idempotent processing, failure handling. Use webhooks when: source supports them, you need real-time, you can expose an endpoint. Use polling when: source doesn't support webhooks, or you need guaranteed delivery with your own schedule."
    },
    {
        "q": "How do you handle API authentication securely in pipelines?",
        "a": "Never hardcode credentials. Use: (1) Cloud secret managers (AWS Secrets Manager, Azure Key Vault). (2) In Airflow: Connections stored encrypted. (3) In Databricks: Secret scopes. (4) For OAuth: implement token refresh (cache token until expiry, refresh automatically on 401). (5) Rotate credentials regularly. (6) Use service accounts (not personal tokens). (7) Audit access to secrets. (8) Different credentials per environment (dev/prod)."
    },
    {
        "q": "How would you build a reverse ETL system?",
        "a": "1) Query Gold layer for current state of entities (segments, scores). 2) Compare with 'last synced' state table (detect changes: new, updated, deleted). 3) For each changed record, call destination API (Salesforce, HubSpot, etc.). 4) Handle rate limits and partial failures (retry queue). 5) Track sync status per record (synced_at, sync_status, error). 6) Alert on failures (> X% error rate). 7) Idempotent: same record synced twice = same result. Tools: Census, Hightouch, or custom Airflow DAG."
    },
]

for i, qa in enumerate(questions, 1):
    print(f"\nQ{i}: {qa['q']}")
    print(f"{'─'*60}")
    print(f"  {qa['a']}")

---
# Summary: APIs & Networking Cheat Sheet

## API Ingestion Decision Matrix

| Scenario | Method | Tool |
|----------|--------|------|
| Simple SaaS extraction | REST + cursor pagination | Fivetran, Airbyte, custom |
| High-volume extraction | Async REST + parallelism | aiohttp, concurrent.futures |
| Real-time events | Webhooks → Kafka | Custom receiver + Kafka |
| Nested/complex data | GraphQL | Custom + graphql-core |
| Internal services | gRPC | grpcio (Python) |
| Push data to SaaS | Reverse ETL | Census, Hightouch, custom |

## The API Ingestion Checklist

1. Authentication: How to get a token? Does it expire?
2. Pagination: Cursor, offset, or keyset? Max page size?
3. Rate limits: What are they? Headers to read?
4. Incremental: Filter by modified_since? Webhook available?
5. Schema: Fixed or evolving? How to handle new fields?
6. Errors: What 4xx/5xx codes? Retry strategy?
7. Idempotency: Can you safely re-run without duplicates?
8. Monitoring: How to know if ingestion stopped working?

---
*APIs & Networking for Data Engineers*